# NPTEL Study Notes: Neural Language Models & Gating Mechanisms

## 1. Recap: Vanilla RNNs and Gradient Instability

In a Vanilla Recurrent Neural Network (RNN), training is performed using **Backpropagation Through Time (BPTT)**. Because BPTT unrolls the network across the time sequence, the gradient of the final loss $J(\theta)$ with respect to an early hidden state $h_1$ requires calculating the product of multiple weight matrices via the chain rule.

This recurrent multiplication leads to two primary issues:

1. **Exploding Gradient Problem:** If the matrix values are large, the gradients grow exponentially. **Solution:** Gradient clipping.
2. **Vanishing Gradient Problem:** If the matrix values are small (e.g., between 0 and 1), the gradients shrink exponentially and eventually vanish. This prevents the network from learning long-term dependencies (e.g., matching a pronoun to a noun mentioned 20 words prior). **Solutions:** Gating mechanisms (LSTM/GRU), Attention mechanisms, or Skip/Residual connections.

---

## 2. Applications of RNNs in NLP

Because RNNs can process sequences of any length, they are highly versatile for downstream NLP tasks.

### 2.1 Sequence Labeling (e.g., POS Tagging, NER)

* **Task:** Predict a label for *every token* in a sequence (e.g., identifying Nouns, Verbs, Adjectives).
* **Mechanism:** At every time step $t$, the hidden state $h_t$ is passed through a linear layer with a weight matrix $U$, followed by a non-linear `softmax` function.
* **Output:** The `softmax` output generates a probability distribution over the possible classes (e.g., 46 POS tags). We sample or select the class with the highest probability.

### 2.2 Sentence Classification (e.g., Sentiment Analysis, Spam Detection)

* **Task:** Predict a single label for the *entire sequence*.
* **Mechanism:**
* *Approach 1:* Process the sequence and take only the **last hidden state**, assuming it has compressed all prior context. Pass it through a linear layer and a non-linearity.
* *Approach 2:* Merge or pool all the hidden states across the sequence into a single vector, then classify.



---

## 3. Advanced RNN Architectures

### Bi-directional RNNs

In tasks like POS tagging or sequence classification where the *entire* input sequence is available at once (unlike text generation), predicting a current word's tag often requires future context as well as past context.

* **Architecture:** Consists of two independent RNNs.
1. A **Forward RNN** reading left-to-right to generate $\vec{h}_t$.
2. A **Backward RNN** reading right-to-left to generate $\overleftarrow{h}_t$.


* **Combination:** At every time step $t$, the two hidden states are concatenated or merged to form a single comprehensive hidden state $h_t = [\vec{h}_t ; \overleftarrow{h}_t]$, which is then passed to the output layer.

### Multi-Layer (Stacked) RNNs

Instead of just passing information horizontally through time, RNNs can be stacked vertically.

* **Mechanism:** The output of the hidden state at layer $l-1$ for time step $t$ becomes the *input* for the hidden state at layer $l$ for time step $t$. This allows the network to learn hierarchical representations.

---

## 4. Addressing Vanishing Gradients: LSTM & GRU

To fix the vanishing gradient problem, explicitly designed architectures were introduced that use **gates**—vectors containing values between 0 and 1 (usually generated via a sigmoid function). These act as valves, mathematically controlling how much information flows through by element-wise multiplication.

### 4.1 Long Short-Term Memory (LSTM)

LSTM introduces a secondary state path. Alongside the standard **hidden state ($h_t$)** which acts as the output, it maintains a **cell state ($C_t$)** which acts as the network's internal memory.

An LSTM uses three dynamic gates, each computed using the current input $x_t$ and the previous hidden state $h_{t-1}$:

1. **Forget Gate ($f_t$):** Controls what parts of the *previous cell state* $C_{t-1}$ to keep versus forget.
* If $f_t = 0$, previous memory is erased. If $f_t = 1$, memory is perfectly preserved.


2. **Input Gate ($i_t$):** Controls how much of the newly generated *cell content* ($\tilde{C}_t$) should be written to the cell state.
3. **Output Gate ($o_t$):** Controls what parts of the updated *cell state* $C_t$ should be exposed as the *current hidden state* $h_t$.

**Mathematical Intuition:**
The new cell content is generated using a $\tanh$ activation:


$$\tilde{C}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$$


The core cell state update is a simple linear addition, which is why gradients flow easily without vanishing:


$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$


*(Note: $\odot$ denotes element-wise multiplication)*

> **Key Takeaway:** If $f_t \approx 1$ and $i_t \approx 0$, the cell state is perfectly copied to the next step, allowing information from faraway tokens to bypass standard recurrent degradation.

### 4.2 Gated Recurrent Unit (GRU)

Proposed as a more computationally efficient alternative to LSTM, the GRU merges the cell state and hidden state into a **single hidden state ($h_t$)**. It reduces the architecture to just two gates.

1. **Update Gate ($u_t$):** Replaces both the forget and input gates of the LSTM. It determines how much of the *previous hidden state* to preserve versus how much of the *new content* to apply.
2. **Reset Gate ($r_t$):** Controls which parts of the previous hidden state are allowed to contribute to generating the new candidate content.

**Mathematical Intuition:**


$$h_t = u_t \odot \tilde{h}_t + (1 - u_t) \odot h_{t-1}$$


Because the GRU balances the old state and new state using $u_t$ and $(1 - u_t)$, it requires fewer parameter matrices (weights) to train compared to LSTM.

* **Rule of Thumb:** Start with LSTM if you have a massive dataset (since it has more parameters to learn). If data is scarce or computational efficiency is a priority, use GRU. Neither is universally "better."

---

## 5. Alternative Architectural Solutions

Gating mechanisms aren't the only way to solve vanishing gradients. Modern deep learning utilizes direct structural connections to provide "shortcuts" for gradients.

### 5.1 Residual (Skip) Connections

Instead of passing input $x$ into a layer and simply outputting $f(x)$, a residual connection adds the original input directly to the layer's output:


$$\text{Output} = f(x) + x$$


This ensures the model never completely "forgets" the input due to deep successive transformations. If the layer fails to learn anything useful, the gradient can still flow perfectly backward through the $+ x$ identity path. (This is heavily used in Transformers).

### 5.2 Dense Networks (DenseNets)

Instead of a layer only connecting to the layer immediately after it, DenseNets connect *every layer to all subsequent layers* in a feed-forward fashion. This ensures maximum information flow and prevents early layers from suffering from vanishing gradients.

### 5.3 Highway Networks

A mixture of residual connections and gating mechanisms. A gate dynamically controls the balance between passing the original input unaltered versus passing it through the transformation function, essentially learning when to use a skip connection.

---

---

## 6. Fundamental Limitation of RNNs

Despite architectural improvements like LSTMs and skip connections, all RNN models suffer from a fundamental flaw: **lack of parallelization**. Because the calculation of the hidden state $h_t$ strictly depends on the calculation of $h_{t-1}$, sequences must be processed sequentially. This computational bottleneck paved the way for architectures like the Transformer.

---

## 7. Practice Questions

**Q1: During BPTT in a Vanilla RNN, which characteristic of the weight matrices directly causes the vanishing gradient problem?**

* **A)** The matrices are too sparse.
* **B)** The matrix values are large, compounding to infinity.
* **C)** The matrix values are small, shrinking exponentially when multiplied repeatedly.
* **D)** The matrices are not symmetric.

> **Answer: C**. Gradients shrink when the sequential multiplication involves matrices with small singular values (e.g., between 0 and 1).

**Q2: In an LSTM, what happens if the Forget Gate vector $f_t$ contains all 1s, and the Input Gate vector $i_t$ contains all 0s?**

* **A)** The cell memory is completely wiped.
* **B)** The previous cell state $C_{t-1}$ is perfectly preserved and copied to $C_t$.
* **C)** The new candidate content $\tilde{C}_t$ completely overwrites the memory.
* **D)** The model experiences exploding gradients.

> **Answer: B**. Looking at the formula $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$, if $f_t=1$ and $i_t=0$, it simplifies to $C_t = C_{t-1}$.

**Q3: Which of the following statements about GRUs is TRUE compared to LSTMs?**

* **A)** GRUs maintain a separate cell state alongside the hidden state.
* **B)** GRUs generally have more parameters to train than LSTMs.
* **C)** GRUs combine the functionality of forget and input gates into a single Update Gate.
* **D)** GRUs are exclusively bi-directional.

> **Answer: C**. GRUs use the Update Gate to control both what is retained from the past and what is added from the present, removing the need for a separate cell state.

**Q4: How does a residual (skip) connection mitigate vanishing gradients?**

* **A)** By replacing standard activation functions with sigmoids.
* **B)** By providing a direct additive pathway ($f(x) + x$) that allows gradients to bypass deep non-linear transformations.
* **C)** By clipping gradients before they reach zero.
* **D)** By using gating mechanisms to control memory.

> **Answer: B**. The $+ x$ term acts as a direct gradient "superhighway," preventing the gradient from vanishing even if the derivative of $f(x)$ becomes extremely small.

---

## 8. Revision Summary

* **Vanilla RNNs:** Suffer from Exploding (fix: clipping) and Vanishing (fix: gates/skip connections) gradients due to BPTT matrix multiplication.
* **Bi-RNNs:** Combine Forward (L->R) and Backward (R->L) RNN states to capture bidirectional context; useful when the whole sequence is available.
* **LSTM:** Mitigates vanishing gradients via **Cell State (Memory)** and 3 gates (**Forget, Input, Output**). Computationally heavy (more parameters).
* **GRU:** Simpler than LSTM; only has a **Hidden State** and 2 gates (**Update, Reset**). Faster to train, works well with less data.
* **Skip/Residual Connections:** Adds input directly to output ($f(x) + x$) to create a clean gradient pathway.
* **DenseNets/Highway Nets:** Architectures that feed layer outputs to all future layers or use learned gates to manage skip connections.
* **Limitation:** RNNs (even LSTMs) cannot process sequences in parallel because step $t$ fundamentally requires the output from step $t-1$.